In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [15]:
data = pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

# Save encoders and scaler for future use
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scalar, file)

In [25]:
## Define a function to create the model and try different parameters(Keras Classifier)

def create_model(neurons=32, layers=1):
    model = Sequential()
    # Use Input layer as first layer instead of input_shape parameter
    model.add(tf.keras.layers.Input(shape=(X_train.shape[1],)))
    model.add(Dense(neurons, activation='relu'))
    
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [26]:
## Create a KerasClassifier for use in scikit-learn
## Keras classifier is responsible for creating entire ANN
model = KerasClassifier(model=create_model, neurons=32, layers=1, epochs=50, batch_size=10, verbose=0)

In [23]:
## Define grid Search parameters
## Use model__ prefix for model parameters in scikeras
param_grid = {
    'model__neurons': [16, 32, 64, 128],
    'model__layers': [1, 2],
    # 'batch_size': [10, 20],
    'epochs': [50, 100]
}

In [27]:
## Perform Grid Search
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1)
grid_result = grid.fit(X_train, y_train)

## Print the best parameters and accuracy
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Best: 0.857875 using {'epochs': 50, 'model__layers': 1, 'model__neurons': 32}
